In [86]:
#Importing necessary Libraries
import pandas as pd
import numpy as np

In [87]:
# Loading the dataset- The imported data set has been data validated(03_data_validation.ipynb)
loan_df = pd.read_excel('../Data/validated_loan_df.xlsx')
comm_df = pd.read_excel('../Data/validated_comm_df.xlsx')

In [88]:
# Create working copy
fe_df = loan_df.copy()
co_df = comm_df.copy()

In [89]:
# Communication Features- Total communication events
total_events = (
    co_df
    .groupby('Loan Id')
    .size()
    .reset_index(name='total_comm_events')
)

In [90]:
# Communication Features- Days Since Last Communication
co_df['Timestamp'] = pd.to_datetime(
    co_df['Timestamp'],
    errors='coerce'
)

last_comm = (
    co_df
    .groupby('Loan Id')['Timestamp']
    .max()
    .reset_index(name='last_comm_timestamp')
)

reference_date = co_df['Timestamp'].max()

last_comm['days_since_last_comm'] = (
    reference_date -
    last_comm['last_comm_timestamp']
).dt.days

In [91]:
# Communication Features- Number of unique communication types
unique_event_types = (
    co_df
    .groupby('Loan Id')['Event Type']
    .nunique()
    .reset_index(name='unique_event_types')
)

In [92]:
# Financial Features-Balance Ratio

fe_df['balance_ratio'] = (
    fe_df['current_balance'] /
    (fe_df['original_balance'] + 1)
)

In [93]:
# Financial Features-Payment Flag

fe_df['payment_flag'] = (
    fe_df['last_pmt_amt'] > 0
).astype(int)

In [94]:
# Communication Effectiveness Features-Connect Rate
fe_df['connect_rate'] = (
    fe_df['times_connect'] /
    (fe_df['times_dials'] + 1)
)

In [95]:
# Communication Effectiveness Features- RPC Rate
fe_df['rpc_rate'] = (
    fe_df['times_rpc'] /
    (fe_df['times_connect'] + 1)
)

In [96]:
# Communication Effectiveness Features- Contact Efficiency
fe_df['contact_efficiency'] = (
    fe_df['times_contact'] /
    (fe_df['times_dials'] + 1)
)

In [97]:
# Communication Effectiveness Features- PTP Flag
fe_df['ptp_flag'] = (
    fe_df['times_ptp'] > 0
).astype(int)

In [98]:
# Communication Effectiveness Features- Portal Engagement Flag
fe_df['portal_engaged'] = (
    fe_df['total_portal_visit'] > 0
).astype(int)

In [99]:
# Log Transform Skewed Features- Useful for imbalance-heavy financial data.
# Only log-transform strictly non-negative variables

fe_df['log_times_dials'] = np.log1p(
    fe_df['times_dials']
)

fe_df['log_last_payment'] = np.log1p(
    fe_df['last_pmt_amt']
)

In [100]:
# Merge Loan Demographic and Communication Activity  
comm_features = total_events.merge(
    last_comm[
        ['Loan Id', 'days_since_last_comm']
    ],
    on='Loan Id',
    how='left'
)

comm_features = comm_features.merge(
    unique_event_types,
    on='Loan Id',
    how='left'
)

fe_df = fe_df.merge(
    comm_features,
    on='Loan Id',
    how='left'
)

In [101]:
# Drop high risk unecessary columns

drop_columns = [
    'Loan Id',
    'chargeoff_date',
    'birthday',
    'last_pmt_date',
    'lastNoticeSent'
]

fe_df = fe_df.drop(
    columns=drop_columns,
    errors='ignore'
)
#This avoids:leakage, identifiers, unstable temporal artifacts

In [102]:
# FINAL DATA QUALITY CHECK BEFORE MODELING

print("SHAPE:")
print(fe_df.shape)

print("\n")

print("MISSING VALUES:")
missing_summary = (
    fe_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

display(missing_summary[missing_summary > 0])

print("\n")

print("DUPLICATE ROWS:")
print(fe_df.duplicated().sum())

print("\n")

print("DATA TYPES:")
display(fe_df.dtypes)

print("\n")

print("CHECK FOR INFINITE VALUES:")

numeric_cols = fe_df.select_dtypes(include=['int64', 'float64']).columns

infinite_counts = (np.isinf(fe_df[numeric_cols]).sum().sort_values(ascending=False))

display(infinite_counts[infinite_counts > 0])

print("\n")

print("TARGET DISTRIBUTION:")

display(fe_df['Payment_Next30Days'].value_counts())

print("\n")

print("COMMUNICATION FEATURE SUMMARY:")

comm_feature_cols = [
    'total_comm_events',
    'days_since_last_comm',
    'unique_event_types'
]

display(
    fe_df[comm_feature_cols]
    .describe()
)

print("\n")

print("MISSING VALUES IN COMMUNICATION FEATURES:")

display(
    fe_df[comm_feature_cols]
    .isnull()
    .sum()
)

SHAPE:
(154849, 31)


MISSING VALUES:


days_since_last_comm    137932
total_comm_events       137932
unique_event_types      137932
customer_age             15193
dtype: int64



DUPLICATE ROWS:
84


DATA TYPES:


clnt_no                     object
original_balance           float64
current_balance            float64
last_pmt_amt               float64
status                       int64
state                       object
Creditor name               object
times_dials                  int64
times_connect                int64
times_contact                int64
times_rpc                    int64
times_ptp                    int64
times_up                     int64
times_drop                   int64
times_lm                     int64
total_portal_visit           int64
Payment_Next30Days           int64
customer_age               float64
days_since_last_payment      int64
balance_ratio              float64
payment_flag                 int64
connect_rate               float64
rpc_rate                   float64
contact_efficiency         float64
ptp_flag                     int64
portal_engaged               int64
log_times_dials            float64
log_last_payment           float64
total_comm_events   



CHECK FOR INFINITE VALUES:


...

Payment_Next30Days
0    154220
1       629
Name: count, dtype: int64



COMMUNICATION FEATURE SUMMARY:


,total_comm_events,days_since_last_comm,unique_event_types
count,16917.000000,16917.000000,16917.000000
mean,6.830998,68.811610,2.586747
std,5.683637,73.788793,1.164166
min,1.000000,0.000000,1.000000
25%,2.000000,14.000000,2.000000
50%,5.000000,40.000000,2.000000
75%,11.000000,99.000000,4.000000
max,31.000000,320.000000,5.000000




MISSING VALUES IN COMMUNICATION FEATURES:


total_comm_events       137932
days_since_last_comm    137932
unique_event_types      137932
dtype: int64

In [103]:
# Attending to data quality issues

# Fill Communication Feature Nulls
comm_feature_cols = [
    'total_comm_events',
    'days_since_last_comm',
    'unique_event_types'
]

fe_df[comm_feature_cols] = (
    fe_df[comm_feature_cols]
    .fillna(0)
)

# Missing values
fe_df['customer_age'] = (
    fe_df['customer_age']
    .fillna(fe_df['customer_age'].median())
)

# Duplicate rows
fe_df = fe_df.drop_duplicates()

In [104]:
# Moving the Target variable at the end of the table
target = fe_df.pop('Payment_Next30Days')

fe_df['Payment_Next30Days'] = target
display(fe_df.head(10))

display(fe_df.info())

,clnt_no,original_balance,current_balance,last_pmt_amt,status,state,Creditor name,times_dials,times_connect,times_contact,...,rpc_rate,contact_efficiency,ptp_flag,portal_engaged,log_times_dials,log_last_payment,total_comm_events,days_since_last_comm,unique_event_types,Payment_Next30Days
0,GALOM1,1070.18,1070.18,0.0,79,OH,FSV Bank,0,0,0,...,0.0,0.0,0,0,0.000000,0.000000,0.0,0.0,0.0,0
1,GALOM1,2358.76,2358.76,0.0,29,AZ,FSV Bank,0,0,0,...,0.0,0.0,0,0,0.000000,0.000000,0.0,0.0,0.0,0
2,GAL102,922.58,922.58,0.0,89,TN,SYR Bank,0,0,0,...,0.0,0.0,0,0,0.000000,0.000000,0.0,0.0,0.0,0
3,GALM12,1026.16,1026.16,0.0,89,CA,VPZ Bank,0,0,0,...,0.0,0.0,0,0,0.000000,0.000000,0.0,0.0,0.0,0
4,GALM12,1284.37,1284.37,0.0,52,NJ,SYR Bank,52,0,0,...,0.0,0.0,0,0,3.970292,0.000000,0.0,0.0,0.0,0
5,GAL102,2316.93,2316.93,0.0,89,RI,SYR Bank,0,0,0,...,0.0,0.0,0,0,0.000000,0.000000,0.0,0.0,0.0,0
6,GALM12,3314.43,3314.43,0.0,88,ME,VPZ Bank,35,0,0,...,0.0,0.0,0,0,3.583519,0.000000,0.0,0.0,0.0,0
7,GALO11,2859.85,2859.85,0.0,18,KS,KYK Bank,0,0,0,...,0.0,0.0,0,0,0.000000,0.000000,0.0,0.0,0.0,0
8,RCS001,882.01,702.01,30.0,18,OR,EVD Bank,0,0,0,...,0.0,0.0,0,0,0.000000,3.433987,0.0,0.0,0.0,1
9,GAL102,1003.20,1003.20,0.0,89,IN,SYR Bank,0,0,0,...,0.0,0.0,0,0,0.000000,0.000000,0.0,0.0,0.0,0


<class 'pandas.core.frame.DataFrame'>
Index: 154765 entries, 0 to 154848
Data columns (total 31 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   clnt_no                  154765 non-null  object 
 1   original_balance         154765 non-null  float64
 2   current_balance          154765 non-null  float64
 3   last_pmt_amt             154765 non-null  float64
 4   status                   154765 non-null  int64  
 5   state                    154765 non-null  object 
 6   Creditor name            154765 non-null  object 
 7   times_dials              154765 non-null  int64  
 8   times_connect            154765 non-null  int64  
 9   times_contact            154765 non-null  int64  
 10  times_rpc                154765 non-null  int64  
 11  times_ptp                154765 non-null  int64  
 12  times_up                 154765 non-null  int64  
 13  times_drop               154765 non-null  int64  
 14  times_lm 

None

In [105]:
# Save final feature engineered dataset

fe_df.to_excel('../Data/final_feature_engineered_data.xlsx',index=False)

print("Feature engineered dataset saved successfully.")

Feature engineered dataset saved successfully.
